# Tutorial 00: Configure, simulate, save, reload and plot

One `ExperimentConfig` declares the specimen and instrument, illumination scan
axes, and output choices. `simulate_experiment(experiment)` runs everything and
writes HDF5. Loading and plotting are separate commands that read only that file.
For a one-command simulation plus summary plot, use `run_experiment(experiment)`.

This normal-incidence example is a 30 nm Co film with an in-plane magnetization
and 10% out-of-plane magnon modulation. All lengths are metres, energies eV and
angles radians; recipe literals are nm. A `period` is a full repeat.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/scattering_calculator").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from scattering_calculator import simulation_pipelines as sim


## 1. Define the complete experiment

Leaving `illumination.energies_eV` and `polarizations` as `None` simulates the
single energy/state from `xray`. Set, for example, `(776., 778., 780.)` and
`("CR", "CL")` to run all six combinations through exactly the same runner.
The detector is defined once. Choose the arrays to save in `outputs.save`.
`overwrite=True` makes rerunning this tutorial replace its own result file.


In [ ]:
experiment = sim.ExperimentConfig(
    # 1. X-RAY SOURCE: eV, photons/s, radians
    xray=sim.XRayConfig(energy=778., photon_flux=1e8, pol="LH",
        linear_polarization_angle=np.deg2rad(20.), coherence_length=(1.,1.)),
    # 2. SIMULATION GRID: (y,x) pixels and metres
    simulation=sim.SimulationConfig(shape=(128,128), real_space_pixel_size=4e-9),
    # 3. SAMPLE: recipe literals nm, slice thickness metres
    sample=sim.SampleConfig(recipe="Co(30)", max_slice_thickness=2e-9, sample_name="Co magnon film"),
    # 4. MAGNETIZATION: shared names; period is a full repeat
    magnetic_pattern=sim.MagneticPatternConfig(pattern_type_method="magnon_wave",
        pattern_config={"period":40e-9, "amplitude":.1, "angle":np.deg2rad(30.), "inplane_angle":0., "phase":0.}),
    # 5. APERTURES: None retains a continuous film
    aperture=sim.FrontApertureConfig(aperture_method=None),
    # 6. ILLUMINATION: profile, direction and scan axes
    illumination=sim.IlluminationConfig(illumination_function="gaussian",
        illumination_config={"center":(0.,0.),"distance":0.,"fwhm":150e-9,"alpha_beam":(0.,0.)},
        energies_eV=None, polarizations=None),
    # 7. SAMPLE PROPAGATION: representation and multislice
    propagation=sim.SamplePropagatorConfig(propagator_method="Jones",
        propagator_config={"propagate":True,"farfield_oversampling":2}),
    # 8. DETECTOR / ACQUISITION: geometry, optional analyzer and sensor response
    detector=sim.DetectorConfig(shape=(256,256), detector_center=(128,128), pixel_size=13.5e-6,
        sample_to_detector_distance=.03, detector_propagation_method="fraunhofer", analyzer_angle=None,
        detector_params={"counts_per_photon":1.,"quantum_efficiency":.8,"readout_noise_average":0.,
                         "readout_noise_sigma":2.,"detector_threshold":64000.,"noise_seed":17},
        measurement_config={"exposure_time":.4,"number_frames":1,"max_counts_per_image":None},
        artifacts_config={"sigma_photon":0.,"camera_seed":31,"average_hot_pixels":0.,"average_cold_pixels":0.,"cosmic_rays_per_second":0.}),
    # 9. BEAMSTOP
    beamstop=sim.BeamstopConfig(bs_method=None),
    # 10. OUTPUTS: select physical observables and optional summary panels
    outputs=sim.OutputConfig(path=ROOT / "outputs/00_experiment_workflow/experiment.h5",
        save=("exit_wave","fft","fft_intensity","detector_ideal","detector_measured"), overwrite=True),
    analysis={},
)


## 2. Run everything and save HDF5

No extra detector or spectrum-specific wrapper is needed.


In [ ]:
results = sim.simulate_experiment(experiment)
print("Saved:", results.path)
print("Energies:", results.energies_eV, "polarizations:", results.polarizations)


## 3. Reload and plot without resimulating

These cells need only the imports and the saved filename in a later session.
Complex fields, raw FFT phases and per-exposure camera q coordinates remain
available for further analysis. Large arrays load only when requested.


In [ ]:
results = sim.load_results(ROOT / "outputs/00_experiment_workflow/experiment.h5")
fig = sim.plot_results(results, energy_index=0, polarization_index=0,
    save_path=results.path.with_suffix(".png"))
plt.show()
exit_wave = results.read("exit_wave")
fft_wave = results.read("fft")
print("Exit / FFT:", exit_wave.shape, fft_wave.shape)
print("Detector series (energy, polarization, y, x):", results.stack("detector_ideal").shape)


## Optional: execute all three stages in one command

```python
results, figure = sim.run_experiment(experiment)
```

This runs physics, saves HDF5, reloads it and saves the configured summary plot.
Use `simulate_experiment` when you want to analyze or plot separately.

See [the setup guide](../docs/experiment_setup.md) for saved observables, units,
scan ordering and HDF5 layout; [Tutorial 18](18_co_magnon_linear_analyzer.ipynb)
compares analyzer angles and the crossed-polarizer control.


## Scan parameter alternatives with the same command

Python lists in scalar parameters request a Cartesian scan:

```python
experiment.sample.recipe = ["Co(20)", "Co(30)"]
experiment.propagation.propagator_method = ["Jones", "Scalar"]
experiment.detector.sample_to_detector_distance = [0.03, 0.06]
experiment.illumination.energies_eV = [778., 790.]
experiment.illumination.polarizations = ["LH", "LV"]
experiment.detector.projection_energy = 778.  # optional fixed detector q convention
results = sim.simulate_experiment(experiment)
```

These alternatives produce 32 exposures. Each physical sample/illumination/model
combination is propagated once and reused at both detector distances. Lists that
already describe one structure (apertures, positions, dimensions, Stokes vectors)
keep their existing meaning; tuples and output selections do not request scans.

`xray.energy` controls material constants and physical propagation. The optional
`detector.projection_energy` controls detector q coordinates and defaults to the
physical energy. Finite-distance diffraction retains the physical wavelength.

```python
print(results.parameter_axes)
wave = results.read("exit_wave", scan_index=1, energy_index=0, polarization_index=0)
fig = sim.plot_results(results, scan_index=1)
```

Parameter combinations precede energy and polarization in file order. Each HDF5
run records its selections and whether sample propagation was reused.
